# CPPI and Drawdown ConstraintsConstant Proportion Portfolio Insurance (CPPI) adjusts risky-asset exposure according to the cushion above a protected floor. This notebook compares static-floor CPPI, buy-and-hold risky wealth and a peak-ratcheting drawdown constraint.

In [ ]:
%matplotlib inlineimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport risk_kit as rkindustries = rk.get_ind_returns()risky_returns = industries.loc['2000':, ['Steel', 'Fin', 'Beer']].dropna()START = 1000FLOOR = 0.80MULTIPLIER = 3SAFE_RATE = 0.03safe_monthly = (1 + SAFE_RATE) ** (1/12) - 1print(f'Backtest observations: {len(risky_returns)}')

## 1. Compounding returnsFor a return series $r_t$, cumulative growth is $\prod_t(1+r_t)-1$. The logarithmic identity makes the same calculation possible using a sum of log returns.

In [ ]:
def compound_product(r):    return (1 + r).prod() - 1def compound_log(r):    return np.expm1(np.log1p(r).sum())comparison = pd.DataFrame({'Product': compound_product(risky_returns), 'Log sum': compound_log(risky_returns)})comparison

## 2. Run standard CPPIAt each step, the cushion is the fraction of current wealth above the floor. Risky exposure is the multiplier times the cushion, clipped to [0, 1]. The remainder is invested in the safe asset.

In [ ]:
cppi = rk.run_cppi(risky_returns, riskfree_rate=SAFE_RATE, m=MULTIPLIER,                    start=START, floor=FLOOR, steps_per_year=12)buy_and_hold = cppi['Risky Wealth']fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)for asset in risky_returns.columns:    axes[0].plot(cppi['Wealth'][asset], label=asset)    axes[1].plot(buy_and_hold[asset], label=asset)    axes[2].plot(cppi['Risky Allocation'][asset], label=asset)axes[0].axhline(START*FLOOR, linestyle='--', color='black', label='Initial floor')axes[0].set_title('CPPI wealth')axes[1].set_title('100% risky buy-and-hold wealth')axes[2].set_title('CPPI risky allocation')for ax in axes: ax.legend(ncol=3); ax.grid(alpha=.25)plt.tight_layout()

## 3. Compare performance statisticsThe strategy does not guarantee the floor in the presence of gaps, discrete rebalancing, transaction costs or model mismatch. Historical realised drawdowns should be examined rather than inferred from the rule alone.

In [ ]:
risky_strategy_returns = cppi['Wealth'].pct_change().dropna()buy_hold_returns = buy_and_hold.pct_change().dropna()print('CPPI')display(rk.summary_stats(risky_strategy_returns, SAFE_RATE, 12))print('Buy and hold')display(rk.summary_stats(buy_hold_returns, SAFE_RATE, 12))

## 4. Dynamic drawdown floorWith a drawdown limit $d$, the floor ratchets to the running wealth peak multiplied by $(1-d)$. This is stricter than a static floor because it rises after new highs.

In [ ]:
drawdown_limit = 0.25cppi_dd = rk.run_cppi(risky_returns, riskfree_rate=SAFE_RATE, m=MULTIPLIER,                      start=START, floor=FLOOR, drawdown=drawdown_limit)asset = 'Fin'ax = cppi_dd['Wealth'][asset].plot(figsize=(10, 5), label='CPPI wealth')cppi_dd['Risky Wealth'][asset].plot(ax=ax, linestyle=':', label='Buy and hold')cppi_dd['Floor'][asset].plot(ax=ax, linestyle='--', label='Ratchet floor')ax.set(title=f'Dynamic drawdown constraint ({drawdown_limit:.0%})', ylabel='Account value')ax.legend(); ax.grid(alpha=.25)plt.tight_layout()

## 5. Optional market-wide stress testA reconstructed market-cap-weighted industry index requires companion files for industry firm counts and average firm size. Those files are not included in this repository snapshot. When supplied under the documented names, this cell runs the additional market-wide stress test.

In [ ]:
try:    market_returns = rk.get_total_market_index_returns().loc['2007':]except FileNotFoundError as exc:    print(f'Optional market-cap stress test skipped: {exc}')else:    market_cppi = rk.run_cppi(market_returns, riskfree_rate=SAFE_RATE, m=MULTIPLIER,                              start=START, floor=FLOOR)    market_cppi['Wealth'].plot(figsize=(10, 5), title='Market-cap-weighted proxy: CPPI vs risky wealth')    market_cppi['Risky Wealth'].plot(style='--')    plt.tight_layout()

## Key takeaways and limitations- CPPI exposure responds to the cushion, not directly to volatility forecasts.- A static floor and a ratcheting drawdown floor are different risk controls.- Discrete rebalancing and market gaps can cause floor breaches.- The backtest excludes trading costs, taxes, market impact and liquidity constraints.